In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import math
import keras
import torch
import tensorflow as tf
from keras.utils import image_dataset_from_directory as loader

### System details

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"Tensorflow version: {tf.__version__}")

print("--------------------------------------------------")
print(f"Using cuda: {torch.cuda.is_available()}")
print(f"Cuda device: {torch.cuda.get_device_name(torch.cuda.current_device())}")

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(f"Device Name: {tf.test.gpu_device_name()}")

### Getting the data

In [ ]:
root_dir = "../PKLot/PKLotSegmented"

ds_train = []
ds_test = []
ds_validation = []

total_days, train_days, test_days, validation_days = (0, 0, 0, 0)
temp = 0

subsets = os.listdir(root_dir)
for subset in subsets:
    climatic_condition = os.listdir(f"{root_dir}/{subset}")
    for weather in climatic_condition:
        dates = os.listdir(f"{root_dir}/{subset}/{weather}")

        total_days += len(dates)

        days = len(dates) - 1
        test_qtd = math.floor(days * .5)
        train_qtd = days - test_qtd

        count = len(dates)
        for index, date in enumerate(dates):
            if count == len(dates):
                ds_validation.append(loader(f"{root_dir}/{subset}/{weather}/{date}", image_size=(224, 224)))
                validation_days += 1
            elif count > test_qtd:
                ds_train.append(loader(f"{root_dir}/{subset}/{weather}/{date}", image_size=(224, 224)))
                train_days += 1
            else:
                ds_test.append(loader(f"{root_dir}/{subset}/{weather}/{date}", image_size=(224, 224)))
                test_days += 1
            
            count -= 1

In [ ]:
final_train = ds_train[0].concatenate(ds_train[1])
for index in range(2, len(ds_train)):
    final_train = final_train.concatenate(ds_train[index])

final_test = ds_test[0].concatenate(ds_test[1])
for index in range(2, len(ds_test)):
    final_test = final_test.concatenate(ds_test[index])

final_validation = ds_validation[0].concatenate(ds_test[1])
for index in range(2, len(ds_validation)):
    final_validation = final_validation.concatenate(ds_validation[index])

In [ ]:
print(f"Dataset days: {total_days}")
print(f"Training days: {train_days} - {final_train}")
print(f"Testing days: {test_days} - {final_test}")
print(f"Validation days: {validation_days} - {final_validation}")

### Transfer learning

In [ ]:
base_model = keras.applications.MobileNetV3Large(weights="imagenet", include_top=False)
base_model.summary()

In [ ]:
base_model.trainable = False
keras_input = keras.Input(shape=(224, 224, 3))

new_layer = base_model(keras_input, training=False)
new_layer = keras.layers.GlobalAveragePooling2D()(new_layer)
outputs = keras.layers.Dense(1)(new_layer)
model = keras.Model(keras_input, outputs)

model.summary(show_trainable=True)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(), loss=keras.losses.BinaryCrossentropy(from_logits=True), metrics=[keras.metrics.BinaryAccuracy()])

print("fitting only new layer")
model.fit(final_train, epochs=2, validation_data=final_validation)

### Fine tuning

In [ ]:
base_model.trainable = True
model.summary(show_trainable=True)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=keras.losses.BinaryCrossentropy(from_logits=True), metrics=[keras.metrics.BinaryAccuracy()])

print("Fitting hole model")
model.fit(final_train, epochs=1, validation_data=final_validation)

### Tests and results

In [ ]:
import numpy as np

# Make a model with 2 layers
layer1 = keras.layers.Dense(3, activation="relu")
layer2 = keras.layers.Dense(3, activation="sigmoid")
model = keras.Sequential([keras.Input(shape=(3,)), layer1, layer2])

# Freeze the first layer
layer1.trainable = False

# Keep a copy of the weights of layer1 for later reference
initial_layer1_weights_values = layer1.get_weights()

# Train the model
model.compile(optimizer="adam", loss="mse")
model.fit(np.random.random((2, 3)), np.random.random((2, 3)))

# Check that the weights of layer1 have not changed during training
final_layer1_weights_values = layer1.get_weights()
np.testing.assert_allclose(
    initial_layer1_weights_values[0], final_layer1_weights_values[0]
)
np.testing.assert_allclose(
    initial_layer1_weights_values[1], final_layer1_weights_values[1]
)